# 01. 종속변수 — 셀별 승차인원 변화 (2024 → 2025)

**원본**: `Analisis3/01_morning_change_7_9_250m_stable.py`

- 대상: 2024·2025년 노선·정류장 구성이 같은 **250m 격자 132개** (`stable_250m_grid_centers_143_…csv`, 파일명은 143이지만 실제 132개)
- 조건: 경기 승차(41) → 서울 하차(11)
- 시간대: **07:00~09:59** (본분석). analysis에서 쓰려고 **종일(00~23시)**도 같이 집계한다
- 방법: 교통카드의 `승차정류장ID`를 셀의 2024·2025년 정류장 ID와 매핑

**원본 대비 수정**
- 원본 결과 파일에서 2025년에 정류장 ID가 바뀐 셀 3개(1572, 12816, 13318)가 2025년 0명으로 들어가 있었다 → 아래에서 대조한다

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

In [2]:
stable = read_csv(STABLE_PATH)
stable["셀 ID"] = pd.to_numeric(stable["셀 ID"], errors="coerce")
stable = stable.dropna(subset=["셀 ID"]).drop_duplicates("셀 ID").copy()
stable["셀 ID"] = stable["셀 ID"].astype(int)

# 정류장 ID → 셀 ID (한 셀에 정류장이 여러 개면 'a | b' 형태)
stop_to_cell = {}
for _, row in stable.iterrows():
    for column in ["2024년 정류장 ID", "2025년 정류장 ID"]:
        for stop_id in str(row[column]).split("|"):
            stop_id = stop_id.strip().removesuffix(".0")
            if stop_id and stop_id.lower() != "nan":
                stop_to_cell[stop_id] = int(row["셀 ID"])
print(f"공급구조 유지 셀: {len(stable)}개, 매핑된 정류장 ID: {len(stop_to_cell)}개")

공급구조 유지 셀: 132개, 매핑된 정류장 ID: 141개


In [3]:
def boarding_by_cell(path, hours):
    data = read_csv(path)
    data["셀 ID"] = data["승차정류장ID"].astype("string").str.strip().map(stop_to_cell)
    ride_datetime = data["ride_dt"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(14)
    data["승차시간"] = pd.to_datetime(ride_datetime, format="%Y%m%d%H%M%S", errors="coerce").dt.hour
    data["승차인원"] = pd.to_numeric(data["utztn_nope"], errors="coerce")
    filtered = data[
        data["셀 ID"].notna()
        & (pd.to_numeric(data["ride_ctpv_cd"], errors="coerce") == GYEONGGI)
        & (pd.to_numeric(data["goff_ctpv_cd"], errors="coerce") == SEOUL)
        & data["승차인원"].notna()
        & data["승차시간"].between(*hours, inclusive="both")
    ]
    return filtered.groupby("셀 ID")["승차인원"].sum()


morning = {year: boarding_by_cell(path, MORNING_HOURS) for year, path in RAW_PATHS.items()}
all_day = {year: boarding_by_cell(path, ALL_DAY_HOURS) for year, path in RAW_PATHS.items()}

result = stable[["셀 ID", "중심점 위도", "중심점 경도", "2024년 정류장 ID", "2025년 정류장 ID"]].copy()
result["2024년 07~09시 승차인원"] = result["셀 ID"].map(morning[2024]).fillna(0)
result["2025년 07~09시 승차인원"] = result["셀 ID"].map(morning[2025]).fillna(0)
result["07~09시 승차인원 증감"] = result["2025년 07~09시 승차인원"] - result["2024년 07~09시 승차인원"]
result["07~09시 승차인원 변화율(%)"] = np.where(
    result["2024년 07~09시 승차인원"] > 0,
    result["07~09시 승차인원 증감"] / result["2024년 07~09시 승차인원"] * 100, np.nan)
result["감소 여부"] = (result["07~09시 승차인원 증감"] < 0).astype(int)
result["2024년 종일 승차인원"] = result["셀 ID"].map(all_day[2024]).fillna(0)
result["2025년 종일 승차인원"] = result["셀 ID"].map(all_day[2025]).fillna(0)
result["종일 승차인원 증감"] = result["2025년 종일 승차인원"] - result["2024년 종일 승차인원"]

result.to_csv(DEPENDENT_PATH, index=False, encoding="utf-8-sig")
print(f"07~09시: {morning[2024].sum():,.0f} → {morning[2025].sum():,.0f}명")
print(f"종일   : {all_day[2024].sum():,.0f} → {all_day[2025].sum():,.0f}명")
print(f"감소 셀: {result['감소 여부'].sum()}개 / {len(result)}개")
print(f"저장: {DEPENDENT_PATH.name}")
result.describe().T.round(1)

07~09시: 6,044 → 4,757명
종일   : 19,884 → 16,300명
감소 셀: 78개 / 132개
저장: y_boarding_change.csv


,count,mean,std,min,25%,50%,75%,max
셀 ID,132.0,8166.6,4488.6,44.0,4101.2,8903.0,11862.8,15474.0
중심점 위도,132.0,37.7,0.1,37.6,37.7,37.7,37.7,37.9
중심점 경도,132.0,126.8,0.0,126.7,126.7,126.8,126.8,126.9
2024년 07~09시 승차인원,132.0,45.8,77.2,0.0,5.8,17.5,43.2,487.0
2025년 07~09시 승차인원,132.0,36.0,64.2,0.0,4.0,16.5,34.2,435.0
07~09시 승차인원 증감,132.0,-9.8,31.4,-145.0,-14.0,-3.0,2.0,119.0
07~09시 승차인원 변화율(%),118.0,7.9,160.5,-100.0,-50.0,-27.5,19.5,1500.0
감소 여부,132.0,0.6,0.5,0.0,0.0,1.0,1.0,1.0
2024년 종일 승차인원,132.0,150.6,272.2,1.0,16.8,67.0,130.5,1990.0
2025년 종일 승차인원,132.0,123.5,253.8,1.0,14.0,56.0,99.8,2262.0


### 원본 결과와 대조

In [4]:
original = read_csv(ORIGINAL_OUTPUT / "dataset_7_9_morning_boarding_change_250m_stable.csv")
compare = original[["셀 ID", "2024년 정류장 ID", "2025년 정류장 ID", "2025년 07~09시 승차인원"]].merge(
    result[["셀 ID", "2025년 07~09시 승차인원"]], on="셀 ID", suffixes=(" (원본)", " (정리본)"))
different = compare[compare["2025년 07~09시 승차인원 (원본)"] != compare["2025년 07~09시 승차인원 (정리본)"]]
print(f"2024년 값이 다른 셀: {(original['2024년 07~09시 승차인원'].values != result['2024년 07~09시 승차인원'].values).sum()}개")
print(f"2025년 값이 다른 셀: {len(different)}개  → 모두 2024·2025 정류장 ID가 바뀐 셀")
different

2024년 값이 다른 셀: 0개
2025년 값이 다른 셀: 3개  → 모두 2024·2025 정류장 ID가 바뀐 셀


,셀 ID,2024년 정류장 ID,2025년 정류장 ID,2025년 07~09시 승차인원 (원본),2025년 07~09시 승차인원 (정리본)
14,1572,4104848,4104849,0.0,4.0
107,12816,4104939,4105040,0.0,17.0
115,13318,4104935,4175759,0.0,16.0


→ 원본은 이 세 셀의 2025년 승차를 0명으로 집계해서 가짜 감소(−4, −17, −16명)가 생겼다.
정리본은 2025년 정류장 ID도 매핑해서 바로잡았다. 2025년 합계 4,720 → 4,757명이고, 원본 `006` 노트북의 공간결합 결과와 일치한다.